# Zorse: 2-Node Cross-Site Experiment

This notebook runs the full Zorse pipeline across two Chameleon sites:

| Node | Site | GPU | Count |
|------|------|-----|-------|
| Master (node 0) | CHI@TACC | P100 | 2 |
| Worker (node 1) | CHI@UC | V100 | 4 |

**Steps:**
1. Provision servers and install dependencies
2. Open firewall ports and set up cross-node SSH
3. Profile GPU layer runtimes on each node
4. Profile cluster bandwidth
5. Run the Zorse planner
6. Train with Zorse

---
## Configuration

In [156]:
import os

tacc_lease_name = "grolar_test"
uc_lease_name   = "grolar_test_v100"

tacc_server_name = "node-zorse-tacc"
uc_server_name   = "node-zorse-uc"

image_name = "CC-Ubuntu24.04-CUDA"

tacc_num_gpus = 2
uc_num_gpus = 4

CONDA = "source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate zorse"

---
## 1. Provision servers

### Server 1: P100 at CHI@TACC

In [157]:
import chi
from chi import context, lease, server

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@TACC")

In [158]:
tacc_lease = lease.get_lease(tacc_lease_name)
tacc_lease.show()

HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>grolar_test</t…

Lease Details:
Name: grolar_test
ID: 97e8fdce-67fa-4225-85e1-fbefc9e94316
Status: ACTIVE
Start Date: 2026-03-08 00:00:00
End Date: 2026-03-08 23:00:00
User ID: 72bc55b11c375d05eaef4ab0d75c14816dd1636ef095721034e0d84274e9fb9e
Project ID: 2c377d00885d402eab6c5dd245fd04f4

Node Reservations:
ID: 8143d9ad-61cd-4ea6-a572-2055795b6141, Status: active, Min: 1, Max: 1

Floating IP Reservations:
ID: 3d084962-1e0e-4949-998c-a232b5932b33, Status: active, Amount: 1

Network Reservations:

Flavor Reservations:

Events:


In [159]:
tacc_server = server.Server(
    tacc_server_name,
    reservation_id=tacc_lease.node_reservations[0]["id"],
    image_name=image_name,
    network_name="sharednet1",
)
tacc_server.submit(idempotent=True)
tacc_server.show(type="widget")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server node-zorse-tacc's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,node-zorse-tacc
Id,5027c7b4-5893-4cc5-8934-82971c5fba22
Status,ACTIVE
Image Name,CC-Ubuntu24.04-CUDA
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.52.0.252 (v4) Type: fixed MAC: 44:a8:42:27:12:29 IP: 129.114.109.224 (v4) Type: floating MAC: 44:a8:42:27:12:29
Network Name,sharednet1
Created At,2026-03-08T03:16:08Z
Keypair,utkarsh_anand_uwaterloo_ca-jupyter
Reservation Id,None
Host Id,922b95f55c37f9b3a5e3dbb0a40a2766525279a0028b7d77086b3fa8


Attribute,node-zorse-tacc
Id,5027c7b4-5893-4cc5-8934-82971c5fba22
Status,ACTIVE
Image Name,CC-Ubuntu24.04-CUDA
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.52.0.252 (v4) Type: fixed MAC: 44:a8:42:27:12:29 IP: 129.114.109.224 (v4) Type: floating MAC: 44:a8:42:27:12:29
Network Name,sharednet1
Created At,2026-03-08T03:16:08Z
Keypair,utkarsh_anand_uwaterloo_ca-jupyter
Reservation Id,8143d9ad-61cd-4ea6-a572-2055795b6141
Host Id,922b95f55c37f9b3a5e3dbb0a40a2766525279a0028b7d77086b3fa8


In [160]:
# tacc_server.wait()
# tacc_server.associate_floating_ip()
tacc_server.refresh()
tacc_server.check_connectivity()

Checking connectivity to 129.114.109.224 port 22.


Connection successful


### Server 2: V100 at CHI@UC

In [161]:
context.choose_site(default="CHI@UC")

In [162]:
uc_lease = lease.get_lease(uc_lease_name)
uc_lease.show()

HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>grolar_test_v1…

Lease Details:
Name: grolar_test_v100
ID: de58db2a-d099-47d7-929c-0f71bd275872
Status: ACTIVE
Start Date: 2026-03-07 21:10:00
End Date: 2026-03-08 18:00:00
User ID: d9599faaccc5ea7f21cbefde8531ded71a7ef73e49c617168a0f675e77bd7455
Project ID: 2aa9e8bec2e9406c9fdb58cf4db7c2c2

Node Reservations:
ID: 69a8357c-d73e-4cf3-be8f-a163df623e65, Status: active, Min: 1, Max: 1

Floating IP Reservations:
ID: 9ea97da6-0853-4b85-a2fd-f2981ce1fabf, Status: active, Amount: 1

Network Reservations:

Flavor Reservations:

Events:


In [163]:
uc_server = server.Server(
    uc_server_name,
    reservation_id=uc_lease.node_reservations[0]["id"],
    image_name=image_name,
    network_name="sharednet1",
)
uc_server.submit(idempotent=True)
uc_server.show(type="widget")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server node-zorse-uc's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,node-zorse-uc
Id,73f7b9ec-f1d9-4701-9798-204e9d8c3a5f
Status,ACTIVE
Image Name,CC-Ubuntu24.04-CUDA
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.140.83.156 (v4) Type: fixed MAC: b8:ce:f6:93:bf:7e IP: 192.5.86.211 (v4) Type: floating MAC: b8:ce:f6:93:bf:7e
Network Name,sharednet1
Created At,2026-03-08T03:32:34Z
Keypair,utkarsh_anand_uwaterloo_ca-jupyter
Reservation Id,None
Host Id,bb8d45ac85d55dafa29ef58f8983d2c70678c00be0adda6d95ed404d


Attribute,node-zorse-uc
Id,73f7b9ec-f1d9-4701-9798-204e9d8c3a5f
Status,ACTIVE
Image Name,CC-Ubuntu24.04-CUDA
Flavor Name,baremetal
Addresses,sharednet1: IP: 10.140.83.156 (v4) Type: fixed MAC: b8:ce:f6:93:bf:7e IP: 192.5.86.211 (v4) Type: floating MAC: b8:ce:f6:93:bf:7e
Network Name,sharednet1
Created At,2026-03-08T03:32:34Z
Keypair,utkarsh_anand_uwaterloo_ca-jupyter
Reservation Id,69a8357c-d73e-4cf3-be8f-a163df623e65
Host Id,bb8d45ac85d55dafa29ef58f8983d2c70678c00be0adda6d95ed404d


In [164]:
# uc_server.wait()
# uc_server.associate_floating_ip()
uc_server.refresh()
uc_server.check_connectivity()

Checking connectivity to 192.5.86.211 port 22.


Connection successful


---
## 2. Open firewall ports and set up SSH

### Firewall (bare-metal uses firewalld)

In [165]:
firewall_cmd = (
    "sudo firewall-cmd --zone=public --add-port=12345-12346/tcp && "
    "sudo firewall-cmd --zone=public --add-port=29400-29500/tcp && "
    "sudo firewall-cmd --runtime-to-permanent && "
    "sudo firewall-cmd --reload && "
    "sudo firewall-cmd --zone=public --list-ports"
)

context.choose_site(default="CHI@TACC")
print("Opening ports on TACC...")
tacc_server.execute(firewall_cmd)

context.choose_site(default="CHI@UC")
print("\nOpening ports on UC...")
uc_server.execute(firewall_cmd)

Opening ports on TACC...


success


success
success
success
12345-12346/tcp 29400-29500/tcp



Opening ports on UC...


success


success
success
success
12345-12346/tcp 29400-29500/tcp


<Result cmd='sudo firewall-cmd --zone=public --add-port=12345-12346/tcp && sudo firewall-cmd --zone=public --add-port=29400-29500/tcp && sudo firewall-cmd --runtime-to-permanent && sudo firewall-cmd --reload && sudo firewall-cmd --zone=public --list-ports' exited=0>

### Cross-node SSH

In [166]:
context.choose_site(default="CHI@TACC")
tacc_server.execute(
    "rm -f ~/.ssh/id_ed25519 ~/.ssh/id_ed25519.pub && "
    'ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519 -N "" -q'
)
tacc_pubkey = tacc_server.execute("cat ~/.ssh/id_ed25519.pub").stdout.strip()
print(f"TACC public key: {tacc_pubkey[:60]}...")

context.choose_site(default="CHI@UC")
uc_server.execute(
    "rm -f ~/.ssh/id_ed25519 ~/.ssh/id_ed25519.pub && "
    'ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519 -N "" -q'
)
uc_pubkey = uc_server.execute("cat ~/.ssh/id_ed25519.pub").stdout.strip()
print(f"UC public key: {uc_pubkey[:60]}...")

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAICvxk6kw64IKC+vPvlpWde1fUM44+s9l0lP5VY8QXrun cc@node-zorse-tacc
TACC public key: ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAICvxk6kw64IKC+vPvlpWde1f...


ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIFmU88OJsWrwR8mDGwuSfhHLzUK6RHCu+ZRHEPkYc+ke cc@node-zorse-uc
UC public key: ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIFmU88OJsWrwR8mDGwuSfhHL...


In [167]:
# Cross-authorize: each node trusts the other + itself
context.choose_site(default="CHI@TACC")
tacc_server.execute(
    f'echo "{uc_pubkey}" >> ~/.ssh/authorized_keys && '
    'cat ~/.ssh/id_ed25519.pub >> ~/.ssh/authorized_keys && '
    'chmod 600 ~/.ssh/authorized_keys'
)

context.choose_site(default="CHI@UC")
uc_server.execute(
    f'echo "{tacc_pubkey}" >> ~/.ssh/authorized_keys && '
    'cat ~/.ssh/id_ed25519.pub >> ~/.ssh/authorized_keys && '
    'chmod 600 ~/.ssh/authorized_keys'
)

<Result cmd='echo "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAICvxk6kw64IKC+vPvlpWde1fUM44+s9l0lP5VY8QXrun cc@node-zorse-tacc" >> ~/.ssh/authorized_keys && cat ~/.ssh/id_ed25519.pub >> ~/.ssh/authorized_keys && chmod 600 ~/.ssh/authorized_keys' exited=0>

In [168]:
context.choose_site(default="CHI@UC")
uc_server.execute(f'echo "{tacc_pubkey}" >> ~/.ssh/authorized_keys')

context.choose_site(default="CHI@TACC")
tacc_server.execute(f'echo "{uc_pubkey}" >> ~/.ssh/authorized_keys')

<Result cmd='echo "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIFmU88OJsWrwR8mDGwuSfhHLzUK6RHCu+ZRHEPkYc+ke cc@node-zorse-uc" >> ~/.ssh/authorized_keys' exited=0>

In [169]:
context.choose_site(default="CHI@TACC")
tacc_server.execute(
    f"ssh-keyscan -H {uc_ip} >> ~/.ssh/known_hosts 2>/dev/null && "
    f"ssh-keyscan -H {tacc_ip} >> ~/.ssh/known_hosts 2>/dev/null && "
    "ssh-keyscan -H localhost >> ~/.ssh/known_hosts 2>/dev/null"
)

context.choose_site(default="CHI@UC")
uc_server.execute(
    f"ssh-keyscan -H {tacc_ip} >> ~/.ssh/known_hosts 2>/dev/null && "
    f"ssh-keyscan -H {uc_ip} >> ~/.ssh/known_hosts 2>/dev/null && "
    "ssh-keyscan -H localhost >> ~/.ssh/known_hosts 2>/dev/null"
)

<Result cmd='ssh-keyscan -H 129.114.109.224 >> ~/.ssh/known_hosts 2>/dev/null && ssh-keyscan -H 192.5.86.211 >> ~/.ssh/known_hosts 2>/dev/null && ssh-keyscan -H localhost >> ~/.ssh/known_hosts 2>/dev/null' exited=0>

### Verify SSH

In [170]:
gpu_query = "nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader"

context.choose_site(default="CHI@TACC")
print("TACC -> UC:")
tacc_server.execute(f"ssh cc@{uc_ip} '{gpu_query}'")
print("\nTACC -> localhost:")
tacc_server.execute(f"ssh cc@localhost '{gpu_query}'")

context.choose_site(default="CHI@UC")
print("\nUC -> TACC:")
uc_server.execute(f"ssh cc@{tacc_ip} '{gpu_query}'")
print("\nUC -> localhost:")
uc_server.execute(f"ssh cc@localhost '{gpu_query}'")

TACC -> UC:
0, Tesla V100-PCIE-32GB, 32768 MiB
1, Tesla V100-PCIE-32GB, 32768 MiB
2, Tesla V100-PCIE-32GB, 32768 MiB
3, Tesla V100-PCIE-32GB, 32768 MiB

TACC -> localhost:
0, Tesla P100-PCIE-16GB, 16384 MiB
1, Tesla P100-PCIE-16GB, 16384 MiB



UC -> TACC:
0, Tesla P100-PCIE-16GB, 16384 MiB
1, Tesla P100-PCIE-16GB, 16384 MiB

UC -> localhost:
0, Tesla V100-PCIE-32GB, 32768 MiB
1, Tesla V100-PCIE-32GB, 32768 MiB
2, Tesla V100-PCIE-32GB, 32768 MiB
3, Tesla V100-PCIE-32GB, 32768 MiB


<Result cmd="ssh cc@localhost 'nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader'" exited=0>

---
## 3. Install dependencies

In [141]:
# install_system = (
#     "sudo apt-get update && sudo apt-get install -y nvtop git && "
#     "wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh && "
#     "bash /tmp/miniconda.sh -b -p $HOME/miniconda3 && "
#     "rm /tmp/miniconda.sh && "
#     "$HOME/miniconda3/bin/conda init bash"
# )

# context.choose_site(default="CHI@TACC")
# print("Installing system packages on TACC...")
# tacc_server.execute(install_system)

# context.choose_site(default="CHI@UC")
# print("\nInstalling system packages on UC...")
# uc_server.execute(install_system)

Installing system packages on TACC...
Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:2 http://nova.clouds.archive.ubuntu.com/ubuntu noble InRelease
Get:3 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://nova.clouds.archive.ubuntu.com/ubuntu noble-backports InRelease
Get:6 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1806 kB]
Get:7 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [1564 kB]
Fetched 3496 kB in 1s (2334 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
nvtop is already the newest version (3.0.2-1).
git is already the newest version (1:2.43.0-1ubuntu7.3).
0 upgraded, 0 newly installed, 0 to remove and 224 not upgraded.


ERROR: File or directory already exists: '/home/cc/miniconda3'
If you want to update an existing installation, use the -u option.


UnexpectedExit: Encountered a bad command exit code!

Command: 'sudo apt-get update && sudo apt-get install -y nvtop git && wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh && bash /tmp/miniconda.sh -b -p $HOME/miniconda3 && rm /tmp/miniconda.sh && $HOME/miniconda3/bin/conda init bash'

Exit code: 1

Stdout: already printed

Stderr: already printed



In [ ]:
setup_zorse = (
    "git clone https://github.com/benson-guo/zorse.git ~/zorse && "
    "source $HOME/miniconda3/etc/profile.d/conda.sh && "
    "conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main && "
    "conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r && "
    "conda create -y -n zorse python=3.12 && "
    "conda activate zorse && "
    "pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121 && "
    "pip install transformers==4.46.2 matplotlib timer kneed scikit-learn pulp pymetis"
)

context.choose_site(default="CHI@TACC")
print("Setting up Zorse on TACC...")
tacc_server.execute(setup_zorse)

context.choose_site(default="CHI@UC")
print("\nSetting up Zorse on UC...")
uc_server.execute(setup_zorse)

In [171]:
verify_cmd = (
    f"{CONDA} && python3 -c \"import torch; "
    "print('PyTorch:', torch.__version__); "
    "print('CUDA:', torch.cuda.is_available()); "
    "print('GPUs:', torch.cuda.device_count()); "
    "[print(f'  [{i}] {torch.cuda.get_device_name(i)}') for i in range(torch.cuda.device_count())]\""
)

context.choose_site(default="CHI@TACC")
print("=== TACC ===")
tacc_server.execute(verify_cmd)

context.choose_site(default="CHI@UC")
print("\n=== UC ===")
uc_server.execute(verify_cmd)

=== TACC ===
PyTorch: 2.5.1+cu121
CUDA: True
GPUs: 2
  [0] Tesla P100-PCIE-16GB
  [1] Tesla P100-PCIE-16GB



=== UC ===
PyTorch: 2.5.1+cu121
CUDA: True
GPUs: 4
  [0] Tesla V100-PCIE-32GB
  [1] Tesla V100-PCIE-32GB
  [2] Tesla V100-PCIE-32GB
  [3] Tesla V100-PCIE-32GB


<Result cmd='source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate zorse && python3 -c "import torch; print(\'PyTorch:\', torch.__version__); print(\'CUDA:\', torch.cuda.is_available()); print(\'GPUs:\', torch.cuda.device_count()); [print(f\'  [{i}] {torch.cuda.get_device_name(i)}\') for i in range(torch.cuda.device_count())]"' exited=0>

---
## 4. Detect network interfaces

In [172]:
context.choose_site(default="CHI@TACC")
tacc_ifname = tacc_server.execute(
    "ip route get 8.8.8.8 | head -1 | awk '{print $5}'"
).stdout.strip()

context.choose_site(default="CHI@UC")
uc_ifname = uc_server.execute(
    "ip route get 8.8.8.8 | head -1 | awk '{print $5}'"
).stdout.strip()

print(f"TACC interface: {tacc_ifname}")
print(f"UC interface:   {uc_ifname}")

eno1


enp94s0f0np0
TACC interface: eno1
UC interface:   enp94s0f0np0


---
## 5. Profile model layer runtimes

Run on each node independently. Measures per-layer forward/backward latencies for each GPU type.

In [202]:
model_name = "deepspeedllamav2_3b"
sequence_length = 512
vocab_size = 49152
global_batch_size = 128
dtype = "float16"
master_port = 12345
warmup_iterations = 4
iterations = 4

In [203]:
context.choose_site(default="CHI@TACC")
print("Profiling on TACC (P100)...")
tacc_server.execute(
    f"{CONDA} && cd ~/zorse && ./profile_models.sh {dtype} {sequence_length} {master_port} {model_name}"
)

Profiling on TACC (P100)...
Profiling deepspeedllamav2_3b
Rank 0 local rank 0 world size 1 p100-pciex16
Rank: 0 Constructing latency estimator for deepspeedllamav2_3b on p100-pciex16
Profiling optimizer step
Average Optimizer Pass Time: 6.722918040000003 ms
Profiling optimizer step
Average Optimizer Pass Time: 3.4005127399999977 ms
Profiling optimizer step
Average Optimizer Pass Time: 2.2558516800000006 ms
Profiling optimizer step
Average Optimizer Pass Time: 1.7156010400000008 ms
Profiling optimizer step
Average Optimizer Pass Time: 1.3950168200000004 ms
Profiling optimizer step
Average Optimizer Pass Time: 1.1534663600000008 ms
Profiling optimizer step
Average Optimizer Pass Time: 0.9713892199999996 ms
Profiling optimizer step
Average Optimizer Pass Time: 0.8432120199999997 ms
{
  "tag": "Post Batch 1",
  "allocated": 1.5444321632385254,
  "max_allocated": 1.965012550354004,
  "max_reserved": 2.06640625,
  "cuda_malloc_retries": 0
}
Batch Size 1 Activation Memory: 0.421 Activation Me

[rank0]:[W308 07:08:44.172863881 ProcessGroupNCCL.cpp:1250] Warning: WARNING: process group has NOT been destroyed before we destruct ProcessGroupNCCL. On normal program exit, the application should call destroy_process_group to ensure that any pending NCCL operations have finished in this process. In rare cases this process can exit before this point and block the progress of another member of the process group. This constraint has always been present,  but this warning has only been added since PyTorch 2.4 (function operator())


<Result cmd='source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate zorse && cd ~/zorse && ./profile_models.sh float16 512 12345 deepspeedllamav2_3b' exited=0>

In [204]:
context.choose_site(default="CHI@UC")
print("Profiling on UC (V100)...")
uc_server.execute(
    f"{CONDA} && cd ~/zorse && ./profile_models.sh {dtype} {sequence_length} {master_port} {model_name}"
)

Profiling on UC (V100)...
Profiling deepspeedllamav2_3b
Rank 0 local rank 0 world size 1 v100-pciex32
Rank: 0 Constructing latency estimator for deepspeedllamav2_3b on v100-pciex32
Profiling optimizer step
Average Optimizer Pass Time: 4.508932779999996 ms
Profiling optimizer step
Average Optimizer Pass Time: 2.2514484399999986 ms
Profiling optimizer step
Average Optimizer Pass Time: 1.5113949999999996 ms
Profiling optimizer step
Average Optimizer Pass Time: 1.138170379999999 ms
Profiling optimizer step
Average Optimizer Pass Time: 0.9239030599999991 ms
Profiling optimizer step
Average Optimizer Pass Time: 0.7783089600000003 ms
Profiling optimizer step
Average Optimizer Pass Time: 0.6533679400000001 ms
Profiling optimizer step
Average Optimizer Pass Time: 0.5855452799999997 ms
{
  "tag": "Post Batch 1",
  "allocated": 1.5441880226135254,
  "max_allocated": 2.005082130432129,
  "max_reserved": 2.11328125,
  "cuda_malloc_retries": 0
}
Batch Size 1 Activation Memory: 0.461 Activation Memor

[rank0]:[W308 07:09:18.798011998 ProcessGroupNCCL.cpp:1250] Warning: WARNING: process group has NOT been destroyed before we destruct ProcessGroupNCCL. On normal program exit, the application should call destroy_process_group to ensure that any pending NCCL operations have finished in this process. In rare cases this process can exit before this point and block the progress of another member of the process group. This constraint has always been present,  but this warning has only been added since PyTorch 2.4 (function operator())


<Result cmd='source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate zorse && cd ~/zorse && ./profile_models.sh float16 512 12345 deepspeedllamav2_3b' exited=0>

### Merge profiling data

The planner runs on TACC and needs latency data for both GPU types.

In [205]:
context.choose_site(default="CHI@TACC")
tacc_server.execute(
    f"scp cc@{uc_ip}:~/zorse/data/model_latencies.json /tmp/uc_latencies.json"
)
tacc_server.execute(
    f"{CONDA} && cd ~/zorse && python3 -c \""
    "import json; "
    "a = json.load(open('data/model_latencies.json')); "
    "b = json.load(open('/tmp/uc_latencies.json')); "
    "[a.setdefault(k, {}).update(v) for k, v in b.items()]; "
    "json.dump(a, open('data/model_latencies.json', 'w'), indent=4, sort_keys=True)\""
)
print("Profiling data merged on TACC")

Profiling data merged on TACC


---
## 6. Profile cluster bandwidth

Create hostfile with both nodes (floating IPs) and run `collect_cluster_info.py`.

In [182]:
hostfile_content = (
    f"{tacc_ip},,/home/cc/miniconda3/etc/profile.d/conda.sh,zorse,/home/cc/zorse,{tacc_ifname}\n"
    f"{uc_ip},,/home/cc/miniconda3/etc/profile.d/conda.sh,zorse,/home/cc/zorse,{uc_ifname}"
)

context.choose_site(default="CHI@TACC")
tacc_server.execute(f'cat > ~/zorse/hostfile << "EOF"\n{hostfile_content}\nEOF')
print("Hostfile:")
tacc_server.execute("cat ~/zorse/hostfile")

Hostfile:
129.114.109.224,,/home/cc/miniconda3/etc/profile.d/conda.sh,zorse,/home/cc/zorse,eno1
192.5.86.211,,/home/cc/miniconda3/etc/profile.d/conda.sh,zorse,/home/cc/zorse,enp94s0f0np0


<Result cmd='cat ~/zorse/hostfile' exited=0>

In [178]:
context.choose_site(default="CHI@TACC")
tacc_server.execute(f"sudo ip addr add {tacc_ip}/32 dev lo || true")
context.choose_site(default="CHI@UC")
uc_server.execute(f"sudo ip addr add {uc_ip}/32 dev lo || true")

Error: ipv4: Address already assigned.


Error: ipv4: Address already assigned.


<Result cmd='sudo ip addr add 192.5.86.211/32 dev lo || true' exited=0>

In [191]:
context.choose_site(default="CHI@TACC")
print("Collecting cluster info...")
tacc_server.execute(
    f"{CONDA} && cd ~/zorse && nohup python3 collect_cluster_info.py "
    "--machine_file hostfile "
    "--ib_disable "
    "--output cluster_info.json "
    "--optimize_internode "
    "> collect_cluster_info.log 2>&1 & echo $!"
)

57504


KeyboardInterrupt: 

In [193]:
context.choose_site(default="CHI@TACC")
tacc_server.execute("cat ~/zorse/collect_cluster_info.log")

<Result cmd='cat ~/zorse/collect_cluster_info.log' exited=0>

In [186]:
context.choose_site(default="CHI@TACC")
tacc_server.execute("sudo apt-get install -y iperf3")

context.choose_site(default="CHI@UC")
uc_server.execute("sudo apt-get install -y iperf3")

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libiperf0 libsctp1
Suggested packages:
  lksctp-tools
The following NEW packages will be installed:
  iperf3 libiperf0 libsctp1
0 upgraded, 3 newly installed, 0 to remove and 91 not upgraded.
Need to get 115 kB of archives.
After this operation, 390 kB of additional disk space will be used.
Get:1 http://nova.clouds.archive.ubuntu.com/ubuntu noble/main amd64 libsctp1 amd64 1.0.19+dfsg-2build1 [9146 B]
Get:2 http://nova.clouds.archive.ubuntu.com/ubuntu noble/universe amd64 libiperf0 amd64 3.16-1build2 [87.1 kB]
Get:3 http://nova.clouds.archive.ubuntu.com/ubuntu noble/universe amd64 iperf3 amd64 3.16-1build2 [19.0 kB]


debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


Fetched 115 kB in 0s (438 kB/s)
Selecting previously unselected package libsctp1:amd64.
(Reading database ... 152000 files and directories currently installed.)
Preparing to unpack .../libsctp1_1.0.19+dfsg-2build1_amd64.deb ...
Unpacking libsctp1:amd64 (1.0.19+dfsg-2build1) ...
Selecting previously unselected package libiperf0:amd64.
Preparing to unpack .../libiperf0_3.16-1build2_amd64.deb ...
Unpacking libiperf0:amd64 (3.16-1build2) ...
Selecting previously unselected package iperf3.
Preparing to unpack .../iperf3_3.16-1build2_amd64.deb ...
Unpacking iperf3 (3.16-1build2) ...
Setting up libsctp1:amd64 (1.0.19+dfsg-2build1) ...
Setting up libiperf0:amd64 (3.16-1build2) ...
Setting up iperf3 (3.16-1build2) ...
debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This fronten

debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype

The processor microcode seems to be up-to-date.

Restarting services...

Service restarts being deferred:
 /etc/needrestart/restart.d/dbus.service
 systemctl restart networkd-dispatcher.service
 systemctl restart systemd-logind.service
 systemctl restart unattended-upgrades.service

No containers need to be restarted.

User sessions running outdated binaries:
 cc @ session #1: login[1985]
 cc @ session #2: login[1969]
 cc @ user manager service: systemd[2276]

No VM guests are running outdated hypervisor (qemu) binaries on this host.


Pending kernel upgrade
----------------------

Newer kernel available

The currently running kernel version is 6.8.0-64-generic which is not the 
expected kernel version 6.8.0-101-generic.

Restarting the system to load the new kernel will not be handled automatically, 
so you should consider rebooting.



Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libiperf0 libsctp1
Suggested packages:
  lksctp-tools
The following NEW packages will be installed:
  iperf3 libiperf0 libsctp1
0 upgraded, 3 newly installed, 0 to remove and 224 not upgraded.
Need to get 115 kB of archives.
After this operation, 390 kB of additional disk space will be used.
Get:1 http://nova.clouds.archive.ubuntu.com/ubuntu noble/main amd64 libsctp1 amd64 1.0.19+dfsg-2build1 [9146 B]
Get:2 http://nova.clouds.archive.ubuntu.com/ubuntu noble/universe amd64 libiperf0 amd64 3.16-1build2 [87.1 kB]
Get:3 http://nova.clouds.archive.ubuntu.com/ubuntu noble/universe amd64 iperf3 amd64 3.16-1build2 [19.0 kB]


debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


Fetched 115 kB in 1s (157 kB/s)
Selecting previously unselected package libsctp1:amd64.
(Reading database ... 113653 files and directories currently installed.)
Preparing to unpack .../libsctp1_1.0.19+dfsg-2build1_amd64.deb ...
Unpacking libsctp1:amd64 (1.0.19+dfsg-2build1) ...
Selecting previously unselected package libiperf0:amd64.
Preparing to unpack .../libiperf0_3.16-1build2_amd64.deb ...
Unpacking libiperf0:amd64 (3.16-1build2) ...
Selecting previously unselected package iperf3.
Preparing to unpack .../iperf3_3.16-1build2_amd64.deb ...
Unpacking iperf3 (3.16-1build2) ...
Setting up libsctp1:amd64 (1.0.19+dfsg-2build1) ...
Setting up libiperf0:amd64 (3.16-1build2) ...
Setting up iperf3 (3.16-1build2) ...
debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This fronten

debconf: unable to initialize frontend: Dialog
debconf: (Dialog frontend will not work on a dumb terminal, an emacs shell buffer, or without a controlling terminal.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype

Running kernel seems to be up-to-date.

The processor microcode seems to be up-to-date.

No services need to be restarted.

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.


<Result cmd='sudo apt-get install -y iperf3' exited=0>

In [187]:
context.choose_site(default="CHI@TACC")
tacc_server.execute("sudo firewall-cmd --zone=public --add-port=5201/tcp")
tacc_server.execute("iperf3 -s -p 5201 -D")  # -D daemonizes

success


<Result cmd='iperf3 -s -p 5201 -D' exited=0>

In [188]:
context.choose_site(default="CHI@UC")
uc_server.execute(f"iperf3 -c {tacc_ip} -p 5201 -t 10 -P 4")

Connecting to host 129.114.109.224, port 5201
[  5] local 10.140.83.156 port 35504 connected to 129.114.109.224 port 5201
[  7] local 10.140.83.156 port 35520 connected to 129.114.109.224 port 5201
[  9] local 10.140.83.156 port 35534 connected to 129.114.109.224 port 5201
[ 11] local 10.140.83.156 port 35540 connected to 129.114.109.224 port 5201
[ ID] Interval           Transfer     Bitrate         Retr  Cwnd
[  5]   0.00-1.00   sec  64.6 MBytes   542 Mbits/sec    0   5.53 MBytes       
[  7]   0.00-1.00   sec  37.0 MBytes   310 Mbits/sec  1184   1.45 MBytes       
[  9]   0.00-1.00   sec  65.2 MBytes   547 Mbits/sec    0   5.50 MBytes       
[ 11]   0.00-1.00   sec  63.0 MBytes   528 Mbits/sec   41   3.85 MBytes       
[SUM]   0.00-1.00   sec   230 MBytes  1.93 Gbits/sec  1225             
- - - - - - - - - - - - - - - - - - - - - - - - -
[  5]   1.00-2.00   sec  83.2 MBytes   698 Mbits/sec    0   5.53 MBytes       
[  7]   1.00-2.00   sec  44.0 MBytes   369 Mbits/sec    0   1.53 MB

<Result cmd='iperf3 -c 129.114.109.224 -p 5201 -t 10 -P 4' exited=0>

In [195]:
context.choose_site(default="CHI@TACC")
tacc_server.execute(
    f"""{CONDA} && python3 << 'PYEOF'
import json
with open("/home/cc/zorse/cluster_info.json", "r") as f:
    d = json.load(f)
d["bandwidth"]["internode_bandwidth"] = {{"0": {{"1": 0.3}}, "1": {{"0": 0.3}}}}
d["bandwidth"]["internode_latency"] = {{"0": {{"1": 0.133}}, "1": {{"0": 0.133}}}}
with open("/home/cc/zorse/cluster_info.json", "w") as f:
    json.dump(d, f, indent=4)
print("Internode bandwidth patched")
PYEOF"""
)

Internode bandwidth patched


<Result cmd='source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate zorse && python3 << \'PYEOF\'\nimport json\nwith open("/home/cc/zorse/cluster_info.json", "r") as f:\n    d = json.load(f)\nd["bandwidth"]["internode_bandwidth"] = {"0": {"1": 0.3}, "1": {"0": 0.3}}\nd["bandwidth"]["internode_latency"] = {"0": {"1": 0.133}, "1": {"0": 0.133}}\nwith open("/home/cc/zorse/cluster_info.json", "w") as f:\n    json.dump(d, f, indent=4)\nprint("Internode bandwidth patched")\nPYEOF' exited=0>

In [197]:
tacc_server.execute("cat ~/zorse/cluster_info.json")

{
    "machines": [
        {
            "machine": "129.114.109.224",
            "num_gpus": 2,
            "gpus": [
                {
                    "index": 0,
                    "name": "p100-pcie-16gb",
                    "memory_total_mib": 16384,
                    "id": "gpu_0",
                    "machine": "129.114.109.224",
                    "global_rank": 0
                },
                {
                    "index": 1,
                    "name": "p100-pcie-16gb",
                    "memory_total_mib": 16384,
                    "id": "gpu_1",
                    "machine": "129.114.109.224",
                    "global_rank": 1
                }
            ],
            "machine_index": 0
        },
        {
            "machine": "192.5.86.211",
            "num_gpus": 4,
            "gpus": [
                {
                    "index": 0,
                    "name": "v100-pciex32",
                    "memory_total_mib": 32768,
                

<Result cmd='cat ~/zorse/cluster_info.json' exited=0>

---
## 7. Run the Zorse planner

In [206]:
context.choose_site(default="CHI@TACC")
tacc_server.execute(
    f"{CONDA} && cd ~/zorse && python3 zorse_planner.py "
    f"--cluster_info_file cluster_info.json "
    f"--machine_file hostfile "
    f"--model_name {model_name} "
    f"--global_batch_size {global_batch_size} "
    f"--sequence_length {sequence_length} "
    "--nccl_ib_disable "
    "--use_agrs_comm_model "
    "--output_file training_config.json"
)

Building GPU graph with connection weights...
Graph has 6 nodes and 15 edges
Graph construction complete.

Performing PyMetis min k-cut partitioning with k=2...
Partition time: 0.00s

Partition Results:
Partition 1: 2 GPUs - p100-pciex16, p100-pciex16
Partition 2: 4 GPUs - v100-pciex32, v100-pciex32, v100-pciex32, v100-pciex32
Total edge cut value: 8

Running Stage 2 optimizer on each partition...
Partition 1: 2 GPUs - [GPUStage2(identifier='p100-pciex16', rank=0), GPUStage2(identifier='p100-pciex16', rank=1)]
Estimated All-Gather latency: 0.012663s
Estimated Reduce-Scatter latency: 0.025326s
Partition 2: 4 GPUs - [GPUStage2(identifier='v100-pciex32', rank=3), GPUStage2(identifier='v100-pciex32', rank=4), GPUStage2(identifier='v100-pciex32', rank=2), GPUStage2(identifier='v100-pciex32', rank=5)]
Estimated All-Gather latency: 0.024914s
Estimated Reduce-Scatter latency: 0.049829s
  Activation tensor: [1, 512, 3200] = 0.0031 GB
  Message size: 3.12 MB, Reference latency: 0.000000 ms
  Int

/home/cc/zorse/comm_model.py:106: RuntimeWarning: divide by zero encountered in scalar divide
  transfer_time = data_per_step / bandwidth_matrix[sender, receiver]


1066207886, minimum GPU memory in group 32.0
	model_params: 5699878400.0, sharded_model_params: 0.0, activations_mem: 419430400.0, optimizer_mem: 8549817600.0, pipeline_mem: 209715200.0
 >>>> Optimizing with interleave degree: 1
Stage 0 has 3 layers, base chunk 3, extra 0
Stage 1 has 23 layers, base chunk 23, extra 0
Group Layer Times: {1: 3212.1303947999995, 2: 504.52619988666754}
Cross Group Compute Ratios: [0.13574732747920223, 0.8642526725207978]
For group 1 : Memory estimated 5.355550765991211, minimum GPU memory in group 16
	model_params: 743462400.0, sharded_model_params: 0.0, activations_mem: 419430400.0, optimizer_mem: 2230387200.0, pipeline_mem: 209715200.0
For group 2 : Memory estimated 15.857001066207886, minimum GPU memory in group 32.0
	model_params: 5699878400.0, sharded_model_params: 0.0, activations_mem: 419430400.0, optimizer_mem: 8549817600.0, pipeline_mem: 209715200.0
Interleave Degree: 1, Optimal Latency: 5608703.014552835
	Layer splits: [3, 23]
 >>>> Optimizing wi

<Result cmd='source $HOME/miniconda3/etc/profile.d/conda.sh && conda activate zorse && cd ~/zorse && python3 zorse_planner.py --cluster_info_file cluster_info.json --machine_file hostfile --model_name deepspeedllamav2_3b --global_batch_size 128 --sequence_length 512 --nccl_ib_disable --use_agrs_comm_model --output_file training_config.json' exited=0>

In [207]:
context.choose_site(default="CHI@TACC")
print("Planner output:")
tacc_server.execute("cat ~/zorse/training_config.json")

Planner output:
{
    "model_name": "deepspeedllamav2_3b",
    "global_batch_size": 128,
    "microbatch_size": 2,
    "sequence_length": 512,
    "vocab_size": 49152,
    "fused_optimizer": true,
    "interleave_degree": 2,
    "latency": 10724.252924266668,
    "autocast_dtype": "float16",
    "reduce_dtype": "float32",
    "pipeline_config": [
        {
            "gpu_ranks": [
                0,
                1
            ],
            "num_microbatches_per_rank": [
                32,
                32
            ],
            "layer_partition": [
                0,
                1
            ],
            "zero_config": [
                [
                    0,
                    1
                ]
            ]
        },
        {
            "gpu_ranks": [
                3,
                4,
                2,
                5
            ],
            "num_microbatches_per_rank": [
                16,
                16,
                16,
               

<Result cmd='cat ~/zorse/training_config.json' exited=0>

### Sync config to UC node

In [208]:
context.choose_site(default="CHI@TACC")
tacc_server.execute(
    f"scp ~/zorse/training_config.json cc@{uc_ip}:~/zorse/training_config.json && "
    f"scp ~/zorse/cluster_info.json cc@{uc_ip}:~/zorse/cluster_info.json && "
    f"scp -r ~/zorse/data cc@{uc_ip}:~/zorse/"
)
print("Config synced to UC")

Config synced to UC


context.choose_site(default="CHI@TACC")
tacc_server.execute("sudo systemctl stop firewalld")
context.choose_site(default="CHI@UC")
uc_server.execute("sudo systemctl stop firewalld")---
## 8. Train with Zorse

Launch the worker on UC in the background, then run the master on TACC in the foreground.

In [246]:
firewall_fix = (
    "sudo firewall-cmd --zone=public --add-port=1024-65535/tcp && "
    "sudo firewall-cmd --runtime-to-permanent && "
    "sudo firewall-cmd --reload"
)

context.choose_site(default="CHI@TACC")
tacc_server.execute(firewall_fix)
context.choose_site(default="CHI@UC")
uc_server.execute(firewall_fix)

success
success
success


success
success
success


<Result cmd='sudo firewall-cmd --zone=public --add-port=1024-65535/tcp && sudo firewall-cmd --runtime-to-permanent && sudo firewall-cmd --reload' exited=0>

In [260]:
master_addr = tacc_ip
master_addr = 12345

def build_zorse_cmd(node_rank, nproc, ifname):
    return (
        f"{CONDA} && cd ~/zorse && "
        f"NCCL_IB_DISABLE=1 "
        f"NCCL_SOCKET_IFNAME={ifname} "
        f"GLOO_SOCKET_IFNAME={ifname} "
        f"torchrun "
        f"--nproc_per_node={nproc} "
        f"--nnodes=2 "
        f"--node_rank={node_rank} "
        f"--master_addr={master_addr} "
        f"--master_port={master_port} "
        f"zorse.py "
        f"--config_file training_config.json "
        f"--zero2_pipeline "
        f"--gloo_p2p "
        f"--offload_model_params "
        f"--optimizer_in_backwards"
    )

In [261]:
context.choose_site(default="CHI@UC")
zorse_uc_cmd = build_zorse_cmd(node_rank=1, nproc=uc_num_gpus, ifname=uc_ifname)
uc_server.execute(
    f"nohup bash -c '{zorse_uc_cmd}' > ~/zorse/zorse_worker.log 2>&1 &"
)
print("Zorse worker launched on UC")

Zorse worker launched on UC


In [259]:
context.choose_site(default="CHI@TACC")
zorse_tacc_cmd = build_zorse_cmd(node_rank=0, nproc=tacc_num_gpus, ifname=tacc_ifname)
tacc_server.execute(
    f"bash -c '{zorse_tacc_cmd}' 2>&1 | tee ~/zorse/zorse_master.log"
)

W0308 07:49:58.899000 60328 site-packages/torch/distributed/run.py:793] 
W0308 07:49:58.899000 60328 site-packages/torch/distributed/run.py:793] *****************************************
W0308 07:49:58.899000 60328 site-packages/torch/distributed/run.py:793] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0308 07:49:58.899000 60328 site-packages/torch/distributed/run.py:793] *****************************************


KeyboardInterrupt: 

In [ ]:
context.choose_site(default="CHI@UC")
print("Zorse worker log:")
uc_server.execute("tail -30 ~/zorse/zorse_worker.log")

---
## 9. View results

In [ ]:
context.choose_site(default="CHI@TACC")
print("=== Master log (last 50 lines) ===")
tacc_server.execute("tail -50 ~/zorse/zorse_master.log")

---
## Cleanup

In [ ]:
# context.choose_site(default="CHI@TACC")
# tacc_server.delete()

# context.choose_site(default="CHI@UC")
# uc_server.delete()